# Systematic Review Automation — Google Colab
**LLMs in Qualitative Data Analysis (2023–2026)**

Run all cells top to bottom. The final cell gives you a public URL to open the UI.

---
**What you need:**
- An OpenAI API key (`sk-...`)
- Your search export files (RIS / CSV / BibTeX / PubMed XML)
- PDF files for full-text screening (optional — upload later via UI)


## Cell 1 — Clone the repository

In [ ]:
import os

REPO_URL = 'https://github.com/qbzdm2024/Zhihong.github.io.git'
BRANCH   = 'claude/systematic-review-automation-oC3BZ'
REPO_DIR = '/content/sysrev'
APP_DIR  = f'{REPO_DIR}/systematic-review'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}

# Set working directory to the app folder for all subsequent cells
%cd {APP_DIR}
print('\nWorking directory:', os.getcwd())

## Cell 2 — Install dependencies

In [ ]:
%%capture install_output
!pip install \
    fastapi==0.115.0 \
    uvicorn[standard] \
    pydantic>=2.7.0 \
    pydantic-settings>=2.3.0 \
    openai>=1.50.0 \
    rispy>=0.9.0 \
    python-multipart \
    python-dotenv \
    pyngrok \
    PyMuPDF

print('Dependencies installed.')

In [ ]:
# Verify key packages loaded
import fastapi, openai, pydantic
print(f'fastapi {fastapi.__version__}  openai {openai.__version__}  pydantic {pydantic.__version__}')
print('All packages OK')

## Cell 3 — Create data directories

In [ ]:
import os

for d in ['data/raw', 'data/deduped', 'data/screened',
          'data/extracted', 'data/output', 'data/pdfs']:
    os.makedirs(d, exist_ok=True)

print('Data directories ready:')
!ls -la data/

## Cell 4 — Upload search export files

Upload your `.ris`, `.csv`, `.bib`, `.json`, or `.xml` search export files.
They will be saved to `data/raw/` automatically.

> **Skip this cell if you haven't exported your search results yet** — you can upload files later and re-run just this cell.

In [ ]:
from google.colab import files
import shutil

print('Select your search export files (.ris / .csv / .bib / .json / .xml):')
uploaded = files.upload()

for fname, content in uploaded.items():
    dest = f'data/raw/{fname}'
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  Saved: {dest} ({len(content):,} bytes)')

print(f'\nFiles in data/raw/:')
!ls -lh data/raw/

## Cell 5 — (Optional) Upload PDF files for full-text screening

PDFs should be named `<record_id>.pdf`. You can also upload them later through the UI.

In [ ]:
from google.colab import files

print('Select PDF files (or press Cancel to skip):')
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        if fname.endswith('.pdf'):
            dest = f'data/pdfs/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  Saved: {dest}')
        else:
            print(f'  Skipped (not PDF): {fname}')
except Exception as e:
    print(f'Skipped PDF upload: {e}')

## Cell 6 — Configure ngrok tunnel

ngrok creates a public HTTPS URL that forwards to your Colab server.

**Get a free ngrok auth token at https://dashboard.ngrok.com/get-started/your-authtoken**
(Free account is sufficient)

In [ ]:
from pyngrok import ngrok, conf
from getpass import getpass

# Paste your ngrok auth token when prompted
NGROK_TOKEN = getpass('Paste your ngrok auth token (hidden): ')
ngrok.set_auth_token(NGROK_TOKEN)
print('ngrok token set.')

## Cell 7 — Start the server

This starts FastAPI + opens the ngrok tunnel. **The public URL will appear below.**

> The server runs in the background. **Do not interrupt this cell** — just let it run and look for the URL output.

In [ ]:
import subprocess
import threading
import time
import sys
import os
from pyngrok import ngrok
from IPython.display import display, HTML

APP_DIR = '/content/sysrev/systematic-review'
PORT = 8000

# Kill any existing server on this port
os.system(f'fuser -k {PORT}/tcp 2>/dev/null || true')
time.sleep(1)

# Start uvicorn in background
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.main:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=APP_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for server to be ready
import urllib.request
for attempt in range(20):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/api/pipeline/status', timeout=2)
        print(f'Server ready (attempt {attempt+1})')
        break
    except Exception:
        pass
else:
    print('WARNING: Server did not respond in 20s — check output below')
    out, _ = server_proc.communicate(timeout=2)
    print(out.decode())

# Open ngrok tunnel
public_url = ngrok.connect(PORT, bind_tls=True).public_url
app_url = f'{public_url}/app'

print('\n' + '='*60)
print(f'  API server:  {public_url}/docs')
print(f'  UI (app):    {app_url}')
print('='*60)
print('\nOpen the URL above in your browser.')
print('The setup wizard will guide you through entering your API key.')

display(HTML(f'''
<div style="margin:16px 0; padding:16px; background:#1a1d27; border:1px solid #6c8eff;
  border-radius:8px; font-family:monospace;">
  <div style="color:#8890aa; font-size:12px; margin-bottom:8px;">OPEN IN BROWSER</div>
  <a href="{app_url}" target="_blank"
    style="color:#6c8eff; font-size:18px; font-weight:bold; text-decoration:none;">
    {app_url}
  </a>
</div>
'''))

## Cell 8 — Run pipeline stages (optional — can also use the UI)

You can run stages directly from Python, or use the **Run Pipeline** panel in the UI.

In [ ]:
# Run via API (equivalent to clicking buttons in the UI)
import requests

BASE = f'http://localhost:{PORT}/api'

def run_stage(stage, limit=None):
    r = requests.post(f'{BASE}/pipeline/run',
                      json={'stage': stage, 'limit': limit})
    print(f'{stage}: {r.json()}')

def status():
    r = requests.get(f'{BASE}/pipeline/status')
    counts = r.json()['prisma_counts']
    print('\n── Pipeline Status ──────────────────')
    for k, v in counts.items():
        print(f'  {k:40s} {v}')
    print()

# Uncomment to run:
# run_stage('import')
# run_stage('dedup')
# run_stage('title_screening', limit=50)  # limit for testing
# status()

## Cell 9 — Save outputs to Google Drive (optional)

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_DEST = '/content/drive/MyDrive/SysRev_LLMs_QDA'
os.makedirs(DRIVE_DEST, exist_ok=True)

# Copy outputs
shutil.copytree(
    '/content/sysrev/systematic-review/data/output',
    f'{DRIVE_DEST}/output',
    dirs_exist_ok=True
)

# Save pipeline state
state_src = '/content/sysrev/systematic-review/data/pipeline_state.jsonl'
if os.path.exists(state_src):
    shutil.copy(state_src, f'{DRIVE_DEST}/pipeline_state.jsonl')

print(f'Saved to Google Drive: {DRIVE_DEST}')
!ls -lh {DRIVE_DEST}/

## Cell 10 — Restore session from Google Drive

Colab sessions reset after ~12 hours. Run this to resume from where you left off.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_SRC = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP_DIR   = '/content/sysrev/systematic-review'

if not os.path.exists(DRIVE_SRC):
    print('No saved session found in Drive. Starting fresh.')
else:
    # Restore output files
    if os.path.exists(f'{DRIVE_SRC}/output'):
        shutil.copytree(f'{DRIVE_SRC}/output', f'{APP_DIR}/data/output', dirs_exist_ok=True)
        print('Restored output files.')

    # Restore pipeline state
    state = f'{DRIVE_SRC}/pipeline_state.jsonl'
    if os.path.exists(state):
        os.makedirs(f'{APP_DIR}/data', exist_ok=True)
        shutil.copy(state, f'{APP_DIR}/data/pipeline_state.jsonl')
        print('Restored pipeline state.')

    # Restore PDFs
    if os.path.exists(f'{DRIVE_SRC}/pdfs'):
        shutil.copytree(f'{DRIVE_SRC}/pdfs', f'{APP_DIR}/data/pdfs', dirs_exist_ok=True)
        print('Restored PDF files.')

    print('\nSession restored. Re-run Cell 7 to restart the server.')

## Cell 11 — Stop the server

In [ ]:
from pyngrok import ngrok
ngrok.kill()
server_proc.terminate()
print('Server stopped.')

---
## Quick Reference

| Task | Where |
|------|-------|
| Enter OpenAI API key | UI → Setup Wizard (first run) or Model Config panel |
| Upload search files | Cell 4 above, or place in `data/raw/` |
| Run pipeline stages | UI → Run Pipeline panel, or Cell 8 |
| Review uncertain records | UI → Human Verification panel |
| Upload PDFs | UI → Full Text Needed panel, or Cell 5 |
| Export evidence table | UI → Export panel, or `GET /api/export/evidence-table/csv` |
| Save progress | Cell 9 (saves to Google Drive) |
| Resume after session reset | Cell 10 (restores from Google Drive) |

**Session timeout:** Free Colab sessions disconnect after ~12 hours of inactivity.  
Always save to Drive before closing (Cell 9).